# VodHunter Live Ingest (Colab)

This notebook starts the repo's polling-based live ingest flow for a single Twitch streamer.

Use a GPU runtime if available: `Runtime -> Change runtime type -> GPU`.

Notes:
- This notebook disables EventSub by default and uses polling.
- Your Postgres database with `pgvector` must be reachable from Colab.
- If the Colab session disconnects, live ingest stops.


In [ ]:
# Optional: clone the repo when running in a fresh Colab session.
# Set REPO_URL to your Git remote before running this cell.

REPO_URL = "https://github.com/manofshad/VodHunter"
REPO_DIR = "/content/VodHunter"

if REPO_URL:
    import os
    if not os.path.exists(REPO_DIR):
        !git clone "$REPO_URL" "$REPO_DIR"
    %cd "$REPO_DIR"
else:
    print("Set REPO_URL if you want the notebook to clone the repository automatically.")


In [ ]:
# Install system and Python dependencies.
!apt-get update -y
!apt-get install -y ffmpeg
!pip install -U pip
!pip install -r backend/requirements.txt
!pip install yt-dlp ipywidgets


In [ ]:
# Runtime configuration.
# Fill these in before running the rest of the notebook.

STREAMER = ""
DATABASE_URL = ""
TWITCH_CLIENT_ID = ""
TWITCH_CLIENT_SECRET = ""

# Colab default: disable EventSub and rely on polling.
TWITCH_EVENTSUB_CALLBACK_URL = ""
TWITCH_EVENTSUB_SECRET = ""

# Optional tuning.
MONITOR_POLL_SECONDS = "30"
LIVE_ARCHIVE_POLL_SECONDS = "15"
LIVE_ARCHIVE_LAG_SECONDS = "120"
INGEST_CHUNK_SECONDS = "60"

assert STREAMER.strip(), "Set STREAMER first"
assert DATABASE_URL.strip(), "Set DATABASE_URL first"
assert TWITCH_CLIENT_ID.strip(), "Set TWITCH_CLIENT_ID first"
assert TWITCH_CLIENT_SECRET.strip(), "Set TWITCH_CLIENT_SECRET first"


In [ ]:
# Export environment variables used by the repo.
import os
import sys
from pathlib import Path

os.environ["DATABASE_URL"] = DATABASE_URL.strip()
os.environ["TWITCH_CLIENT_ID"] = TWITCH_CLIENT_ID.strip()
os.environ["TWITCH_CLIENT_SECRET"] = TWITCH_CLIENT_SECRET.strip()
os.environ["TWITCH_EVENTSUB_CALLBACK_URL"] = TWITCH_EVENTSUB_CALLBACK_URL.strip()
os.environ["TWITCH_EVENTSUB_SECRET"] = TWITCH_EVENTSUB_SECRET.strip()
os.environ["MONITOR_POLL_SECONDS"] = MONITOR_POLL_SECONDS.strip()
os.environ["LIVE_ARCHIVE_POLL_SECONDS"] = LIVE_ARCHIVE_POLL_SECONDS.strip()
os.environ["LIVE_ARCHIVE_LAG_SECONDS"] = LIVE_ARCHIVE_LAG_SECONDS.strip()
os.environ["INGEST_CHUNK_SECONDS"] = INGEST_CHUNK_SECONDS.strip()

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")


In [ ]:
# Bootstrap the same live ingest stack used by the admin app.
from backend import bootstrap_admin, bootstrap_ingest, bootstrap_shared
import torch

bootstrap_shared.prepare_runtime_dirs()
store = bootstrap_shared.build_store_state()["store"]
embedder = bootstrap_ingest.build_ingest_state()["embedder"]
monitor_stack = bootstrap_admin.build_monitor_stack(store=store, embedder=embedder)
monitor_manager = monitor_stack["monitor_manager"]
session_query = monitor_stack["session_query"]

print("CUDA available:", torch.cuda.is_available())
print("Monitor ready")


In [ ]:
# Start monitoring and live ingest for the configured streamer.
status = monitor_manager.start(STREAMER)
print(status)


In [ ]:
# Watch current live ingest status.
# Stop this cell manually when you no longer want continuous updates.
import time
from pprint import pprint

REFRESH_SECONDS = 10

try:
    while True:
        status = monitor_manager.get_status()
        print("-" * 80)
        pprint(status)
        time.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    print("Stopped watch loop")


In [ ]:
# Inspect recent live sessions written to the database.
rows = session_query.list_live_sessions(limit=10, offset=0)
for row in rows:
    print(row)


In [ ]:
# Stop the monitor cleanly.
stopped = monitor_manager.stop()
print("Stopped:", stopped)
print(monitor_manager.get_status())
